In [0]:
import requests
import boto3
import json
from datetime import datetime

# --- Config ---
ACCESS_KEY = "ACCESS_KEY"
SECRET_KEY = "SECRET_KEY"
BUCKET = "weather-pipeline-bachi"
REGION = "ap-south-1"

CITIES = {
    "Hyderabad": (17.385, 78.4867),
    "Delhi": (28.6139, 77.2090),
    "Mumbai": (19.0760, 72.8777),
    "Bangalore": (12.9716, 77.5946),
    "Chennai": (13.0827, 80.2707),
}

s3 = boto3.client(
    "s3",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION
)

def fetch_weather(lat, lon):
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&hourly=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m"
        f"&timezone=auto"
    )
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

def run():
    now = datetime.utcnow()
    date_str = now.strftime("%Y-%m-%d")
    hour_str = now.strftime("%H")

    for city, (lat, lon) in CITIES.items():
        data = fetch_weather(lat, lon)
        data["_city"] = city
        data["_ingested_at"] = now.isoformat()

        key = f"bronze/date={date_str}/hour={hour_str}/{city}.json"
        s3.put_object(
            Bucket=BUCKET,
            Key=key,
            Body=json.dumps(data)
        )
        print(f"Wrote {key}")

run()

/home/spark-728c0715-2228-47b1-bd39-32/.ipykernel/71/command-5306624184624469-3249809575:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


Wrote bronze/date=2026-09-23/hour=04/Hyderabad.json
Wrote bronze/date=2026-09-23/hour=04/Delhi.json
Wrote bronze/date=2026-09-23/hour=04/Mumbai.json
Wrote bronze/date=2026-09-23/hour=04/Bangalore.json
Wrote bronze/date=2026-09-23/hour=04/Chennai.json


In [0]:
import boto3
import json
from pyspark.sql import Row
from pyspark.sql.functions import to_timestamp

ACCESS_KEY = "ACCESS_KEY"
SECRET_KEY = "SECRET_KEY"
BUCKET = "weather-pipeline-bachi"
REGION = "ap-south-1"

s3 = boto3.client(
    "s3",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION
)

# List all bronze files
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="bronze/")
keys = [obj["Key"] for obj in response.get("Contents", [])]
print(f"Found {len(keys)} bronze files")

# Read and flatten each file into rows
rows = []
for key in keys:
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    data = json.loads(obj["Body"].read())

    city = data.get("_city")
    hourly = data.get("hourly", {})
    times = hourly.get("time", [])
    temps = hourly.get("temperature_2m", [])
    humidity = hourly.get("relative_humidity_2m", [])
    precip = hourly.get("precipitation", [])
    wind = hourly.get("wind_speed_10m", [])

    for i in range(len(times)):
        rows.append(Row(
            city=city,
            timestamp=times[i],
            temperature_c=temps[i] if i < len(temps) else None,
            humidity_pct=humidity[i] if i < len(humidity) else None,
            precipitation_mm=precip[i] if i < len(precip) else None,
            wind_speed_kmh=wind[i] if i < len(wind) else None
        ))

# Build Spark DataFrame from the rows
raw_df = spark.createDataFrame(rows)
silver_df = raw_df.withColumn("timestamp", to_timestamp("timestamp")) \
                   .dropDuplicates(["city", "timestamp"])

print(f"Silver rows after dedupe: {silver_df.count()}")
silver_df.show(10)

# Write as a managed Delta table (no S3 path needed — Databricks handles storage)
silver_df.write.format("delta").mode("overwrite").saveAsTable("weather_silver")

print("Saved as Delta table: weather_silver")

Found 105 bronze files
Silver rows after dedupe: 960
+---------+-------------------+-------------+------------+----------------+--------------+
|     city|          timestamp|temperature_c|humidity_pct|precipitation_mm|wind_speed_kmh|
+---------+-------------------+-------------+------------+----------------+--------------+
|Bangalore|2026-09-24 11:00:00|         25.8|          62|             0.0|          19.8|
|Bangalore|2026-09-25 03:00:00|         20.8|          91|             0.0|          15.7|
|Bangalore|2026-09-25 11:00:00|         26.3|          64|             0.4|          20.0|
|Bangalore|2026-09-25 23:00:00|         21.1|          88|             0.0|          14.7|
|Bangalore|2026-09-27 10:00:00|         25.4|          62|             0.0|           8.2|
|Bangalore|2026-09-27 14:00:00|         29.2|          47|             0.1|           7.3|
|Bangalore|2026-09-27 16:00:00|         28.7|          46|             0.0|           7.8|
|Bangalore|2026-09-27 23:00:00|      

In [0]:
from pyspark.sql.functions import to_date, avg, max as spark_max, min as spark_min, sum as spark_sum, round as spark_round

gold_df = spark.sql("""
    SELECT
        city,
        DATE(timestamp) as date,
        ROUND(AVG(temperature_c), 1) as avg_temp_c,
        ROUND(MAX(temperature_c), 1) as max_temp_c,
        ROUND(MIN(temperature_c), 1) as min_temp_c,
        ROUND(AVG(humidity_pct), 1) as avg_humidity_pct,
        ROUND(SUM(precipitation_mm), 2) as total_precip_mm,
        ROUND(AVG(wind_speed_kmh), 1) as avg_wind_kmh
    FROM weather_silver
    GROUP BY city, DATE(timestamp)
    ORDER BY city, date
""")

gold_df.show(20, truncate=False)

# Save as a Gold Delta table
gold_df.write.format("delta").mode("overwrite").saveAsTable("weather_gold_daily_summary")

print(f"Gold table saved: {gold_df.count()} rows")

+---------+----------+----------+----------+----------+----------------+---------------+------------+
|city     |date      |avg_temp_c|max_temp_c|min_temp_c|avg_humidity_pct|total_precip_mm|avg_wind_kmh|
+---------+----------+----------+----------+----------+----------------+---------------+------------+
|Bangalore|2026-09-23|23.2      |26.5      |20.0      |69.3            |0.4            |16.7        |
|Bangalore|2026-09-24|22.4      |28.2      |19.6      |80.0            |5.8            |16.8        |
|Bangalore|2026-09-25|22.9      |27.8      |20.6      |81.8            |4.8            |15.1        |
|Bangalore|2026-09-26|23.7      |28.8      |20.0      |70.7            |0.5            |13.4        |
|Bangalore|2026-09-27|23.9      |29.2      |18.0      |70.3            |0.3            |7.1         |
|Bangalore|2026-09-28|24.7      |30.0      |20.0      |68.8            |3.6            |4.8         |
|Bangalore|2026-09-29|23.7      |28.5      |20.4      |75.3            |12.6      

In [0]:
from pyspark.sql.functions import col, count, when, isnan, max as spark_max, current_timestamp, datediff

print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

# 1. Row count per city (completeness check)
print("\n--- Row count per city ---")
spark.sql("SELECT city, COUNT(*) as row_count FROM weather_silver GROUP BY city ORDER BY city").show()

# 2. Null checks on key fields
print("\n--- Null value counts ---")
null_counts = spark.sql("""
    SELECT
        SUM(CASE WHEN temperature_c IS NULL THEN 1 ELSE 0 END) as null_temp,
        SUM(CASE WHEN humidity_pct IS NULL THEN 1 ELSE 0 END) as null_humidity,
        SUM(CASE WHEN precipitation_mm IS NULL THEN 1 ELSE 0 END) as null_precip,
        SUM(CASE WHEN wind_speed_kmh IS NULL THEN 1 ELSE 0 END) as null_wind
    FROM weather_silver
""")
null_counts.show()

# 3. Range / sanity checks — flag impossible values
print("\n--- Sanity check violations ---")
violations = spark.sql("""
    SELECT city, timestamp, temperature_c, humidity_pct, precipitation_mm
    FROM weather_silver
    WHERE humidity_pct > 100 OR humidity_pct < 0
       OR temperature_c < -10 OR temperature_c > 55
       OR precipitation_mm < 0
""")
violation_count = violations.count()
print(f"Found {violation_count} rows violating sanity ranges")
if violation_count > 0:
    violations.show()

# 4. Freshness check — is the latest ingested data recent?
print("\n--- Freshness check ---")
freshness = spark.sql("""
    SELECT city, MAX(timestamp) as latest_reading
    FROM weather_silver
    GROUP BY city
    ORDER BY city
""")
freshness.show(truncate=False)

# 5. Duplicate check (should be 0 since we deduped in silver)
print("\n--- Duplicate check ---")
dup_count = spark.sql("""
    SELECT city, timestamp, COUNT(*) as cnt
    FROM weather_silver
    GROUP BY city, timestamp
    HAVING COUNT(*) > 1
""").count()
print(f"Duplicate (city, timestamp) pairs: {dup_count}")

print("\n" + "=" * 60)
print("DATA QUALITY CHECK COMPLETE")
print("=" * 60)

# Automated pass/fail gate
if violation_count > 0 or dup_count > 0:
    raise Exception(f"❌ Data quality check FAILED: {violation_count} sanity violations, {dup_count} duplicates")
else:
    print("✅ All data quality checks PASSED")

DATA QUALITY REPORT

--- Row count per city ---
+---------+---------+
|     city|row_count|
+---------+---------+
|Bangalore|      240|
|  Chennai|      240|
|    Delhi|      240|
|Hyderabad|      240|
|   Mumbai|      240|
+---------+---------+


--- Null value counts ---
+---------+-------------+-----------+---------+
|null_temp|null_humidity|null_precip|null_wind|
+---------+-------------+-----------+---------+
|        0|            0|          0|        0|
+---------+-------------+-----------+---------+


--- Sanity check violations ---
Found 0 rows violating sanity ranges

--- Freshness check ---
+---------+-------------------+
|city     |latest_reading     |
+---------+-------------------+
|Bangalore|2026-10-02 23:00:00|
|Chennai  |2026-10-02 23:00:00|
|Delhi    |2026-10-02 23:00:00|
|Hyderabad|2026-10-02 23:00:00|
|Mumbai   |2026-10-02 23:00:00|
+---------+-------------------+


--- Duplicate check ---
Duplicate (city, timestamp) pairs: 0

DATA QUALITY CHECK COMPLETE
✅ All data